In [1]:
# 第一次运行如果缺少这些包，把下一行前面的 # 删除
%pip install pandas numpy requests pyarrow matplotlib

import os
# os 用来读取环境变量，例如你的 MASSIVE_API_KEY

import time
# time 用来控制 API 请求间隔，避免超过 Massive 免费限速

import getpass
# getpass 可以安全输入 API key，不会直接显示在屏幕上

from pathlib import Path
# Path 用来创建和管理本地文件夹，比手写字符串路径更方便

import requests
# requests 用来向 Massive API 发 HTTP 请求

import numpy as np
# numpy 用来处理数值计算和缺失值

import pandas as pd
# pandas 是整个分析系统的核心，用来处理时间序列和 DataFrame

import matplotlib.pyplot as plt
# matplotlib 后面用来画价格、成交量和事件图


START = "2026-05-01"
# 我们的数据开始日期
# 之所以从 5/1 开始，是因为你提到 140 左右的早期支撑位

END = "2026-08-22"
# 研究截止日期

TICKERS = ["NBIS", "QQQ", "SPY", "CRWV"]
# NBIS 是主研究对象
# QQQ 是科技股 benchmark
# SPY 是大盘 benchmark
# CRWV 是 AI infrastructure peer

TIMEZONE = "America/New_York"
# 所有时间最后都转换成美东时间
# 这样 9:30、16:00、CPI 8:30 等时间才方便分析

MIN_SECONDS_BETWEEN_CALLS = 12.5
# Massive 免费版约 5 calls/min
# 理论上 60 / 5 = 12 秒
# 我们用 12.5 秒，稍微留一点安全余量


DATA_DIR = Path("data/raw")
# 创建一个路径对象，表示原始数据保存在哪里。data/raw 只是提前创建好的本地文件夹，并不代表数据已经保存在里面。
# API 下载的数据通常先临时存放在 Jupyter 内存（RAM）的 df 中；只有运行 df.to_csv("data/raw/NBIS_data.csv", index=False)，
# 才会把内存中的数据写入 SSD 硬盘，并真正保存到电脑的 data/raw 文件夹里，关闭 Jupyter、重启 Kernel 或关机后也不会消失。

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# parents=True：
# 如果 data 文件夹不存在，也一起创建
#
# exist_ok=True：
# 文件夹已经存在也不会报错


API_KEY = os.getenv("MASSIVE_API_KEY")
# 先看看电脑环境变量里面有没有保存 API key

if not API_KEY:
    # 如果没有找到 API key

    API_KEY = getpass.getpass(
        "Enter your Massive API key: "
    )
    # 让你手动输入
    # 输入内容不会显示出来

Note: you may need to restart the kernel to use updated packages.


In [2]:
BASE_URL = "https://api.massive.com"
# Massive REST API 的基础网址


_last_call_time = 0
# 用来记录上一次 API 请求是什么时候
# 第一次还没请求，所以设置为 0


def massive_get(url, params=None, max_retries=5):
    # 定义一个函数
    #
    # url：
    # 要请求的 Massive 地址
    #
    # params：
    # API 参数，例如 API key
    #
    # max_retries：
    # 如果网络出错，最多重新试 5 次

    global _last_call_time
    # 告诉 Python：
    # 我要修改函数外面的 _last_call_time

    for attempt in range(max_retries):
        # 最多尝试 max_retries 次

        elapsed = time.time() - _last_call_time
        # 计算距离上一次 API 请求已经过去多少秒

        if (
            _last_call_time != 0
            and elapsed < MIN_SECONDS_BETWEEN_CALLS
        ):
            # 如果不是第一次请求
            # 并且距离上次请求不到 12.5 秒

            wait_time = (
                MIN_SECONDS_BETWEEN_CALLS
                -
                elapsed
            )
            # 算出还需要等多久

            time.sleep(wait_time)
            # 程序暂停，避免 Massive rate limit


        try:
            # 开始尝试请求 API

            response = requests.get(
                url,
                params=params,
                timeout=60
            )
            # requests.get：
            # 向 Massive 请求数据
            #
            # timeout=60：
            # 最多等 60 秒


            _last_call_time = time.time()
            # 请求发出后记录当前时间


            if response.status_code == 429:
                # HTTP 429 = 请求太频繁

                print(
                    "Rate limit reached. "
                    "Waiting 60 seconds..."
                )

                time.sleep(60)
                # 等一分钟

                continue
                # 回到 for loop，再试一次


            if response.status_code >= 500:
                # 500 系列通常是服务器暂时故障

                wait = 2 ** attempt
                # exponential backoff
                #
                # 第一次：1秒
                # 第二次：2秒
                # 第三次：4秒
                # 第四次：8秒

                print(
                    f"Server error "
                    f"{response.status_code}. "
                    f"Retrying in {wait}s..."
                )

                time.sleep(wait)

                continue


            response.raise_for_status()
            # 如果是 400、401、403 等错误
            # 这里会直接抛出异常


            return response
            # 如果一切正常，返回 Massive response


        except requests.RequestException as error:
            # 如果是断网、timeout 等 requests 错误

            if attempt == max_retries - 1:
                # 如果已经是最后一次尝试

                raise error
                # 正式报错，不再继续


            wait = 2 ** attempt
            # 否则等一会再尝试

            print(
                f"Network error: {error}. "
                f"Retrying in {wait}s..."
            )

            time.sleep(wait)


    raise RuntimeError(
        "Massive request failed."
    )
    # 理论上前面应该已经 return 或 raise
    # 这一行是额外保险

In [3]:
def cache_path(ticker):
    # 给每只股票生成自己的缓存文件名

    filename = (
        f"{ticker}_1min_"
        f"{START}_"
        f"{END}.parquet"
    )
    # 例如：
    #
    # NBIS_1min_2026-05-01_2026-08-22.parquet

    return DATA_DIR / filename
    # 最后路径类似：
    #
    # data/raw/NBIS_1min_2026-05-01_2026-08-22.parquet

In [4]:
def download_minute_data(
    ticker,
    start=START,
    end=END
):
    # ticker 例如 NBIS
    # start/end 是下载时间范围


    url = (
        f"{BASE_URL}/v2/aggs/ticker/"
        f"{ticker}/range/1/minute/"
        f"{start}/{end}"
    )
    # Massive aggregates endpoint
    #
    # ticker = NBIS
    # multiplier = 1
    # timespan = minute
    #
    # 意思就是：
    # 给我 NBIS 每一分钟的 aggregate bar


    params = {
        "adjusted": "true",
        # 使用经过 corporate action adjustment 的数据，比如拆股

        "sort": "asc",
        # 按时间从早到晚排列

        "limit": 50000,
        # 每次最多要求 50,000 个 bars，一页，超过的去到下一页

        "apiKey": API_KEY
        # Massive API key
    }


    results = []
    # 创建一个空 list
    # 每一页 API 返回的数据都会放进去


    page = 1
    # 当前是第几页


    while url:
        # 只要还有 url
        # 就继续下载下一页

        print(
            f"{ticker}: "
            f"downloading page {page}"
        )


        response = massive_get(
            url,
            params=params
        )
        # 使用刚才写的限速请求函数


        payload = response.json()
        # Massive 返回 JSON
        # 转换成 Python dictionary


        page_results = payload.get(
            "results",
            []
        )
        # 取出这一页真正的 market data
        #
        # 如果没有 results
        # 默认使用空 list


        results.extend(page_results)
        # 把这一页的数据追加到总结果里面


        next_url = payload.get(
            "next_url"
        )
        # Massive 如果数据还有下一页
        # 会返回 next_url


        if next_url:
            # 如果确实有下一页

            url = next_url
            # 下一轮请求 next_url

            params = {
                "apiKey": API_KEY
            }
            # next_url 通常已经包含其他参数
            # 我们只重新附 API key

            page += 1
            # 页码 +1


        else:
            # 如果没有 next_url

            url = None
            # while loop 结束


    if not results:
        # 如果最后什么数据都没拿到

        raise ValueError(
            f"No data returned for {ticker}"
        )


    df = pd.DataFrame(results)
    # 把 Massive 的 JSON list
    # 转换成 pandas DataFrame


    rename_map = {
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
        "n": "transactions"
    }
    # Massive 为了节省数据
    # API 使用 t/o/h/l/c/v/vw/n 等缩写
    #
    # 我们改成人类容易理解的名字


    df = df.rename(
        columns=rename_map
    )
    # 正式改列名


    df["datetime"] = pd.to_datetime(
        df["timestamp"],
        unit="ms",
        utc=True
    )
    # Massive timestamp 是 Unix milliseconds
    #
    # 把它变成 pandas datetime
    #
    # utc=True：
    # 先告诉 pandas 原始时间是 UTC


    df["datetime"] = (
        df["datetime"]
        .dt
        .tz_convert(TIMEZONE)
    )
    # UTC → America/New_York
    #
    # 例如：
    # 13:30 UTC
    # →
    # 09:30 New York


    df["ticker"] = ticker
    # 给每一行加股票代码


    df = (
        df
        .sort_values("datetime")
        # 按时间排序

        .drop_duplicates(
            subset="datetime"
        )
        # 如果 Massive 意外返回重复 bar
        # 删除重复时间

        .set_index("datetime")
        # 把 datetime 变成 DataFrame index
    )


    return df
    # 返回最终整理好的分钟数据

In [5]:
def load_or_download(
    ticker,
    force_download=False
):
    # ticker：
    # 要读取哪只股票
    #
    # force_download=False：
    # 默认优先使用本地缓存
    #
    # 如果以后真的想重新下载
    # 可以设置 True


    path = cache_path(ticker)
    # 找到这只股票应该保存在哪里


    if (
        path.exists()
        and not force_download
    ):
        # 如果文件已经存在
        # 并且你没有要求强制重新下载

        print(
            f"{ticker}: "
            f"loading local cache"
        )


        df = pd.read_parquet(path)
        # 直接读取本地 Parquet


        return df
        # 返回数据
        #
        # 到这里完全没有调用 Massive


    print(
        f"{ticker}: "
        f"cache not found, downloading..."
    )
    # 本地没有，才开始下载


    df = download_minute_data(
        ticker
    )
    # 从 Massive 下载


    df.to_parquet(
        path
    )
    # 保存成 Parquet


    print(
        f"{ticker}: "
        f"saved to {path}"
    )


    return df
    # 返回数据

In [6]:
bars = {}
# 创建 dictionary
#
# 最后会长这样：
#
# bars["NBIS"]
# bars["QQQ"]
# bars["SPY"]
# bars["CRWV"]


for ticker in TICKERS:
    # 依次处理四个 ticker

    bars[ticker] = load_or_download(
        ticker
    )
    # 有缓存 → 本地读取
    #
    # 没缓存 → Massive 下载 + 保存


    print(
        ticker,
        len(bars[ticker]),
        "minute bars"
    )
    # 显示下载/读取了多少根一分钟 K 线

NBIS: cache not found, downloading...
NBIS: downloading page 1
NBIS: downloading page 2
NBIS: saved to data\raw\NBIS_1min_2026-05-01_2026-08-22.parquet
NBIS 64272 minute bars
QQQ: cache not found, downloading...
QQQ: downloading page 1
QQQ: downloading page 2
QQQ: saved to data\raw\QQQ_1min_2026-05-01_2026-08-22.parquet
QQQ 73094 minute bars
SPY: cache not found, downloading...
SPY: downloading page 1
SPY: downloading page 2
SPY: saved to data\raw\SPY_1min_2026-05-01_2026-08-22.parquet
SPY 70273 minute bars
CRWV: cache not found, downloading...
CRWV: downloading page 1
CRWV: downloading page 2
CRWV: saved to data\raw\CRWV_1min_2026-05-01_2026-08-22.parquet
CRWV 65253 minute bars


In [7]:
import os
print(os.getcwd())

c:\Users\rz321\AppData\Local\Programs\Microsoft VS Code


AppData 是 Windows 默认隐藏文件夹。最简单的打开方式：

按键盘 Win + R
输入：
%LOCALAPPDATA%
按 Enter

它会直接打开：

C:\Users\rz321\AppData\Local

然后依次进入：

Programs
→ Microsoft VS Code
→ data
→ raw

In [8]:
def add_session_columns(df):
    # 输入一个 minute DataFrame

    df = df.copy()
    # 复制一份
    # 避免直接修改原始数据


    minute_of_day = (
        df.index.hour * 60
        +
        df.index.minute
    )
    # 把时间转换成“一天中的第几分钟”
    #
    # 例如：
    #
    # 9:30
    # =
    # 9×60 + 30
    # =
    # 570


    conditions = [
        (
            minute_of_day >= 4 * 60
        )
        &
        (
            minute_of_day
            <
            9 * 60 + 30
        ),
        # 04:00–09:29
        # Premarket


        (
            minute_of_day
            >=
            9 * 60 + 30
        )
        &
        (
            minute_of_day
            <
            16 * 60
        ),
        # 09:30–15:59
        # Regular session


        (
            minute_of_day
            >=
            16 * 60
        )
        &
        (
            minute_of_day
            <
            20 * 60
        )
        # 16:00–19:59
        # After-hours
    ]


    labels = [
        "premarket",
        "regular",
        "afterhours"
    ]
    # 上面三个条件分别叫什么


    df["session"] = np.select(
        conditions,
        labels,
        default="other"
    )
    # 给每根 minute bar
    # 标记它属于哪个 session


    df["date"] = pd.to_datetime(
        df.index.date
    )
    # 单独生成交易日期
    #
    # 方便以后：
    # df[df["date"] == 某一天]


    return df


for ticker in TICKERS:
    # 给所有股票都做同样处理

    bars[ticker] = add_session_columns(
        bars[ticker]
    )

In [9]:
def get_window_return(
    regular_session,
    minutes
):
    # 计算开盘后 N 分钟收益率


    if regular_session.empty:
        # 如果当天没有正常交易数据

        return np.nan


    start_price = (
        regular_session.iloc[0]["open"]
    )
    # 9:30 第一根 bar 的 open


    start_time = (
        regular_session.index[0]
    )
    # 当天第一根 regular bar 的时间


    cutoff = (
        start_time
        +
        pd.Timedelta(
            minutes=minutes
        )
    )
    # 例如 minutes=30
    #
    # 9:30 + 30min
    # =
    # 10:00


    window = regular_session[
        regular_session.index < cutoff
    ]
    # 取：
    #
    # 9:30 ≤ time < 10:00


    if window.empty:
        # 没数据

        return np.nan


    end_price = (
        window.iloc[-1]["close"]
    )
    # 这个时间窗口最后一根 bar 的 close


    return (
        end_price / start_price - 1
    )
    # 返回收益率
    # #算这个是为了看：某个事件发生后，股票开盘后的最初反应有多强、方向是什么。

In [10]:
def create_daily_summary(df):
    # 输入完整一分钟数据
    # 输出每天一行


    rows = []
    # 每天的结果暂时存到这里


    trading_dates = sorted(
        df["date"].unique()
    )
    # 找出所有交易日期并排序


    for date in trading_dates:
        # 一天一天分析


        day = df[
            df["date"] == date
        ]
        # 把某一天所有分钟数据取出来


        pre = day[
            day["session"]
            ==
            "premarket"
        ]
        # 当天盘前


        reg = day[
            day["session"]
            ==
            "regular"
        ]
        # 当天正常交易


        aft = day[
            day["session"]
            ==
            "afterhours"
        ]
        # 当天盘后


        if reg.empty:
            # 如果当天没有正常交易数据

            continue
            # 跳到下一天


        open_price = (
            reg.iloc[0]["open"]
        )
        # 正常交易第一根 bar 的 open


        close_price = (
            reg.iloc[-1]["close"]
        )
        # 正常交易最后一根 bar 的 close


        high_time = (
            reg["high"].idxmax()
        )
        # 找出当天最高价出现在哪一分钟


        low_time = (
            reg["low"].idxmin()
        )
        # 找出最低价出现在哪一分钟


        day_high = reg.loc[
            high_time,
            "high"
        ]
        # 当天 high


        day_low = reg.loc[
            low_time,
            "low"
        ]
        # 当天 low


        total_volume = (
            reg["volume"].sum()
        )
        # 正常交易时间全天总 volume


        if (
            "vwap" in reg.columns
            and reg["vwap"].notna().any()
            and total_volume > 0
        ):
            # 如果 Massive 给了 VWAP 数据

            day_vwap = np.average(
                reg["vwap"].fillna(
                    reg["close"]
                ),
                weights=reg["volume"]
            )
            # 用 volume 加权一分钟 VWAP

        else:
            # 如果没有 vwap 字段

            day_vwap = np.average(
                reg["close"],
                weights=reg["volume"]
            )
            # 用一分钟 close 做近似 VWAP


        row = {
            "date": pd.Timestamp(date),
            # 日期

            "open": open_price,
            # 9:30 open

            "high": day_high,
            # 当天最高

            "high_time": high_time,
            # 当天最高出现时间

            "low": day_low,
            # 当天最低

            "low_time": low_time,
            # 当天最低出现时间

            "close": close_price,
            # 16:00 close

            "volume": total_volume,
            # 全天成交量

            "vwap": day_vwap,
            # 当天 VWAP

            "first_15m_return":
                get_window_return(
                    reg,
                    15
                ),
            # 开盘前15分钟收益

            "first_30m_return":
                get_window_return(
                    reg,
                    30
                ),
            # 前30分钟

            "first_60m_return":
                get_window_return(
                    reg,
                    60
                )
            # 前60分钟
        }


        if not pre.empty:
            # 如果当天有盘前交易

            row[
                "premarket_open"
            ] = pre.iloc[0]["open"]

            row[
                "premarket_high"
            ] = pre["high"].max()

            row[
                "premarket_low"
            ] = pre["low"].min()

            row[
                "premarket_close"
            ] = pre.iloc[-1]["close"]

            row[
                "premarket_volume"
            ] = pre["volume"].sum()

        else:
            # 没有盘前数据

            row["premarket_open"] = np.nan

            row["premarket_high"] = np.nan

            row["premarket_low"] = np.nan

            row["premarket_close"] = np.nan

            row["premarket_volume"] = np.nan


        if not aft.empty:
            # 如果有盘后

            row[
                "afterhours_close"
            ] = aft.iloc[-1]["close"]

            row[
                "afterhours_volume"
            ] = aft["volume"].sum()

        else:

            row[
                "afterhours_close"
            ] = np.nan

            row[
                "afterhours_volume"
            ] = np.nan


        rows.append(row)
        # 把这一天放进结果


    daily = pd.DataFrame(rows)
    # 所有天转换成 DataFrame


    daily = (
        daily
        .sort_values("date")
        .set_index("date")
    )
    # 按日期排序
    # 日期作为 index


    daily["previous_close"] = (
        daily["close"].shift(1)
    )
    # 前一个交易日 close


    daily["premarket_return"] = (
        daily["premarket_close"]
        /
        daily["previous_close"]
        -
        1
    )
    # 昨天收盘 → 今天盘前最后价格


    daily["open_gap"] = (
        daily["open"]
        /
        daily["previous_close"]
        -
        1
    )
    # 昨天 close → 今天 9:30 open


    daily["daily_return"] = (
        daily["close"]
        /
        daily["previous_close"]
        -
        1
    )
    # 昨天 close → 今天 close


    daily["intraday_return"] = (
        daily["close"]
        /
        daily["open"]
        -
        1
    )
    # 今天 9:30 open → close
    #
    # 这和 daily_return 不一样


    daily["range_pct"] = (
        daily["high"]
        -
        daily["low"]
    ) / daily["open"]
    # 当天 high-low range
    # 相对于开盘价格


    price_range = (
        daily["high"]
        -
        daily["low"]
    )
    # 先计算每天价格范围


    daily["clv"] = np.where(
        price_range > 0,

        (
            daily["close"]
            -
            daily["low"]
        )
        /
        price_range,

        0.5
    )
    # CLV = Close Location Value
    #
    # 接近 1：
    # close 靠近全天 high
    #
    # 接近 0：
    # close 靠近全天 low


    prior_average_volume = (
        daily["volume"]
        .rolling(
            20,
            min_periods=5
        )
        .mean()
        .shift(1)
    )
    # 计算之前20天平均成交量
    #
    # shift(1) 很重要
    # 防止把今天自己的 volume
    # 放入 benchmark


    daily["rvol20"] = (
        daily["volume"]
        /
        prior_average_volume
    )
    # Relative Volume
    #
    # 2.0 =
    # 今天成交量是过去平均的2倍


    daily["close_vs_vwap"] = (
        daily["close"]
        /
        daily["vwap"]
        -
        1
    )
    # close 是否站在 VWAP 上方


    daily["afterhours_return"] = (
        daily["afterhours_close"]
        /
        daily["close"]
        -
        1
    )
    # 正常收盘 → 盘后最后价格


    return daily

In [11]:
indicator_explanation = pd.DataFrame({
    "指标": [
        "open",
        "high / low",
        "high_time / low_time",
        "close",
        "vwap",
        "first_15m_return",
        "first_30m_return",
        "first_60m_return",
        "premarket_return",
        "open_gap",
        "daily_return",
        "intraday_return",
        "range_pct",
        "clv",
        "rvol20",
        "close_vs_vwap",
        "afterhours_return"
    ],

    "含义": [
        "正常交易第一根 bar（9:30）的开盘价",
        "正常交易时段的最高价和最低价",
        "最高价和最低价出现的一分钟 bar",
        "15:59 bar 的收盘价，即接近 16:00 的正常收盘价",
        "正常交易时段的成交量加权平均价格",
        "9:30 开盘价 → 9:44 bar 收盘价",
        "9:30 开盘价 → 9:59 bar 收盘价",
        "9:30 开盘价 → 10:29 bar 收盘价",
        "前一交易日 16:00 收盘 → 今天盘前最后价格",
        "前一交易日 16:00 收盘 → 今天 9:30 开盘价",
        "前一交易日 16:00 收盘 → 今天 16:00 收盘",
        "今天 9:30 开盘 → 今天 16:00 收盘",
        "（当天最高价 − 当天最低价）÷ 当天开盘价",
        "收盘价处于当天最高价和最低价之间的位置",
        "当天成交量 ÷ 此前 20 个交易日的平均成交量",
        "当天收盘价相对当天 VWAP 的百分比差异",
        "今天 16:00 正常收盘 → 当天盘后最后价格"
    ],

    "计算公式": [
        "第一根 regular bar 的 open",
        "regular high 最大值 / low 最小值",
        "high.idxmax() / low.idxmin()",
        "最后一根 regular bar 的 close",
        "分钟 VWAP 按成交量加权",
        "9:44 close / 9:30 open - 1",
        "9:59 close / 9:30 open - 1",
        "10:29 close / 9:30 open - 1",
        "premarket_close / previous_close - 1",
        "open / previous_close - 1",
        "close / previous_close - 1",
        "close / open - 1",
        "(high - low) / open",
        "(close - low) / (high - low)",
        "volume / 前 20 日平均 volume",
        "close / vwap - 1",
        "afterhours_close / close - 1"
    ]
})

display(indicator_explanation)

,指标,含义,计算公式
0,open,正常交易第一根 bar（9:30）的开盘价,第一根 regular bar 的 open
1,high / low,正常交易时段的最高价和最低价,regular high 最大值 / low 最小值
2,high_time / low_time,最高价和最低价出现的一分钟 bar,high.idxmax() / low.idxmin()
3,close,15:59 bar 的收盘价，即接近 16:00 的正常收盘价,最后一根 regular bar 的 close
4,vwap,正常交易时段的成交量加权平均价格,分钟 VWAP 按成交量加权
5,first_15m_return,9:30 开盘价 → 9:44 bar 收盘价,9:44 close / 9:30 open - 1
6,first_30m_return,9:30 开盘价 → 9:59 bar 收盘价,9:59 close / 9:30 open - 1
7,first_60m_return,9:30 开盘价 → 10:29 bar 收盘价,10:29 close / 9:30 open - 1
8,premarket_return,前一交易日 16:00 收盘 → 今天盘前最后价格,premarket_close / previous_close - 1
9,open_gap,前一交易日 16:00 收盘 → 今天 9:30 开盘价,open / previous_close - 1


# Close Location Value
# Relative Volume over a 20-day period

In [12]:
daily = {}

for ticker in TICKERS:
    daily[ticker] = create_daily_summary(
        bars[ticker]
    )

# 取出 NBIS 的结果
nbis_daily = daily["NBIS"]

# 创建一个只用于显示的副本
nbis_daily_display = nbis_daily.copy()

# 只显示小时和分钟，去掉日期、秒和 -04:00
nbis_daily_display["high_time"] = (
    nbis_daily_display["high_time"]
    .dt.strftime("%H:%M")
)

nbis_daily_display["low_time"] = (
    nbis_daily_display["low_time"]
    .dt.strftime("%H:%M")
)

# 显示前5行
display(nbis_daily_display.head())

,open,high,high_time,low,low_time,close,volume,vwap,first_15m_return,first_30m_return,...,previous_close,premarket_return,open_gap,daily_return,intraday_return,range_pct,clv,rvol20,close_vs_vwap,afterhours_return
date,,,,,,,,,,,,,,,,,,,,,
2026-05-01,141.330,156.00,12:01,140.00,09:31,154.45,1.438801e+07,151.588122,0.051617,0.068492,...,NaN,NaN,NaN,NaN,0.092832,0.113210,0.903125,NaN,0.018879,0.003561
2026-05-04,160.385,179.40,15:39,159.70,09:30,176.31,2.260992e+07,172.712039,0.066012,0.098825,...,154.45,0.039107,0.038427,0.141534,0.099292,0.122829,0.843147,NaN,0.020832,-0.005161
2026-05-05,171.650,179.96,10:22,168.71,09:31,175.80,9.940083e+06,175.936279,0.013458,0.010195,...,176.31,-0.027338,-0.026431,-0.002893,0.024177,0.065540,0.630222,NaN,-0.000775,0.001138
2026-05-06,179.595,195.99,14:35,175.11,10:18,194.98,1.720124e+07,188.430424,0.007823,-0.009883,...,175.80,0.022753,0.021587,0.109101,0.085665,0.116262,0.951628,NaN,0.034759,-0.008616
2026-05-07,190.010,197.89,10:33,177.00,12:39,184.78,1.697735e+07,186.123340,-0.020873,-0.008789,...,194.98,-0.025028,-0.025490,-0.052313,-0.027525,0.109942,0.372427,NaN,-0.007217,-0.029711


In [13]:
events = pd.DataFrame([
    # 创建事件 DataFrame


    {
        "date": "2026-05-12",
        "time": "08:30",
        "event": "CPI",
        "category": "Macro"
    },


    {
        "date": "2026-05-13",
        "time": "08:00",
        "event": "NBIS Q1 Earnings",
        "category": "Company"
    },


    {
        "date": "2026-06-10",
        "time": "08:30",
        "event": "CPI",
        "category": "Macro"
    },


    {
        "date": "2026-06-17",
        "time": "14:00",
        "event": "FOMC",
        "category": "Macro"
    },


    {
        "date": "2026-07-14",
        "time": "08:30",
        "event": "CPI",
        "category": "Macro"
    },


    {
        "date": "2026-07-29",
        "time": "14:00",
        "event": "FOMC",
        "category": "Macro"
    },


    {
        "date": "2026-08-12",
        "time": "08:00",
        "event": "NBIS Q2 Earnings",
        "category": "Company"
    },


    {
        "date": "2026-08-12",
        "time": "08:30",
        "event": "CPI",
        "category": "Macro"
    }
])


events["timestamp"] = pd.to_datetime(
    events["date"]
    +
    " "
    +
    events["time"]
)
# date + time
# 例如：
#
# 2026-08-12 08:30


events["timestamp"] = (
    events["timestamp"]
    .dt
    .tz_localize(TIMEZONE)
)
# 告诉 pandas：
# 这些时间全部是 New York time


events
# 显示事件表

,date,time,event,category,timestamp
0,2026-05-12,08:30,CPI,Macro,2026-05-12 08:30:00-04:00
1,2026-05-13,08:00,NBIS Q1 Earnings,Company,2026-05-13 08:00:00-04:00
2,2026-06-10,08:30,CPI,Macro,2026-06-10 08:30:00-04:00
3,2026-06-17,14:00,FOMC,Macro,2026-06-17 14:00:00-04:00
4,2026-07-14,08:30,CPI,Macro,2026-07-14 08:30:00-04:00
5,2026-07-29,14:00,FOMC,Macro,2026-07-29 14:00:00-04:00
6,2026-08-12,08:00,NBIS Q2 Earnings,Company,2026-08-12 08:00:00-04:00
7,2026-08-12,08:30,CPI,Macro,2026-08-12 08:30:00-04:00


In [14]:
def price_before(df, timestamp):
    # 找事件之前最后一个成交价格


    previous = df[
        df.index < timestamp
    ]


    if previous.empty:
        return np.nan


    return previous.iloc[-1]["close"]

In [15]:
def price_after(
    df,
    timestamp,
    minutes
):
    # 找事件 N 分钟后的价格


    target = (
        timestamp
        +
        pd.Timedelta(
            minutes=minutes
        )
    )


    future = df[
        df.index >= target
    ]


    if future.empty:
        return np.nan


    return future.iloc[0]["close"]

In [16]:
def event_reaction(
    ticker,
    timestamp
):
    # 分析某只股票某一个事件


    df = bars[ticker]
    # 取得一分钟数据


    event_date = pd.Timestamp(
        timestamp.date()
    )
    # 取得事件日期


    day = df[
        df["date"]
        ==
        event_date
    ]
    # 只看当天


    if day.empty:
        return {}


    before = price_before(
        day,
        timestamp
    )
    # 事件前最后价格


    result = {
        "ticker": ticker,
        "pre_event_price": before
    }


    for minutes in [
        1,
        5,
        15,
        30,
        60,
        120
    ]:
        # 分析：
        #
        # 1分钟
        # 5分钟
        # 15分钟
        # 30分钟
        # 60分钟
        # 120分钟


        after = price_after(
            day,
            timestamp,
            minutes
        )


        if (
            pd.notna(before)
            and pd.notna(after)
        ):

            result[
                f"return_{minutes}m"
            ] = (
                after / before - 1
            )

        else:

            result[
                f"return_{minutes}m"
            ] = np.nan


    return result

In [17]:
def price_after(day, timestamp, minutes):
    """
    返回事件发生 N 分钟后的最后一根可用 minute bar 的 close。

    day:
        事件当天的一分钟数据

    timestamp:
        事件时间，例如 2026-06-17 14:00:00-04:00

    minutes:
        事件后多少分钟，例如 5、15、30、60、120
    """

    # 确保数据按时间排列
    day = day.sort_index()

    # 事件后的目标时间
    target_time = (
        timestamp
        + pd.Timedelta(minutes=minutes)
    )

    # 取事件发生后，到目标时间为止的数据
    window = day[
        (day.index >= timestamp)
        & (day.index <= target_time)
    ]

    # 如果这段时间没有交易数据
    if window.empty:
        return np.nan

    # 返回目标时间之前最后一根可用 bar 的收盘价
    return window.iloc[-1]["close"]

In [18]:
event_rows = []
# 保存每一个事件的结果


for _, event in events.iterrows():
    # 一行一行读取 event table

    row = {
        "date": event["date"],
        "event": event["event"],
        "category": event["category"],
        "timestamp": event["timestamp"]
    }

    for ticker in TICKERS:
        # 同一个事件
        # 同时检查 NBIS / QQQ / SPY / CRWV

        reaction = event_reaction(
            ticker,
            event["timestamp"]
        )

        for key, value in reaction.items():

            if key != "ticker":

                row[
                    f"{ticker}_{key}"
                ] = value

    event_rows.append(row)


# 转成最终 DataFrame
event_study = pd.DataFrame(
    event_rows
)


# ============================================================
# 创建仅用于显示的副本
# ============================================================

event_study_display = event_study.copy()


# timestamp只显示日期、小时和分钟
# 例如：2026-05-12 08:30
event_study_display["timestamp"] = (
    event_study_display["timestamp"]
    .dt.strftime("%Y-%m-%d %H:%M")
)


# 找出所有列名中包含 return 的列
return_columns = [
    column
    for column in event_study_display.columns
    if "return" in column.lower()
]


# 将所有return列显示成百分比
# 例如：
# 0.0516  →  5.16%
# -0.0273 → -2.73%
# 找出所有return列
return_columns = [
    column
    for column in event_study_display.columns
    if "return" in column.lower()
]

# 把return列转换成百分比文字
for column in return_columns:
    event_study_display[column] = (
        event_study_display[column].map(
            lambda value: f"{value:.2%}"
            if pd.notna(value)
            else ""
        )
    )

display(event_study_display)

,date,event,category,timestamp,NBIS_pre_event_price,NBIS_return_1m,NBIS_return_5m,NBIS_return_15m,NBIS_return_30m,NBIS_return_60m,...,SPY_return_30m,SPY_return_60m,SPY_return_120m,CRWV_pre_event_price,CRWV_return_1m,CRWV_return_5m,CRWV_return_15m,CRWV_return_30m,CRWV_return_60m,CRWV_return_120m
0,2026-05-12,CPI,Macro,2026-05-12 08:30,182.0000,-0.42%,-0.03%,0.01%,0.33%,-0.91%,...,0.19%,0.07%,-0.25%,112.0500,-0.22%,0.04%,0.40%,0.23%,-2.62%,-6.50%
1,2026-05-13,NBIS Q1 Earnings,Company,2026-05-13 08:00,207.7000,0.83%,1.35%,2.24%,-0.09%,-0.21%,...,-0.21%,-0.23%,-0.37%,112.2158,0.25%,0.68%,0.43%,-0.19%,-0.66%,-4.06%
2,2026-06-10,CPI,Macro,2026-06-10 08:30,210.7500,2.49%,2.79%,2.85%,2.02%,2.52%,...,0.23%,0.26%,0.74%,94.8400,0.62%,1.34%,1.75%,1.33%,1.96%,4.86%
3,2026-06-17,FOMC,Macro,2026-06-17 14:00,290.7600,-0.65%,-2.97%,0.09%,-0.41%,1.85%,...,-0.66%,-0.57%,-1.20%,120.3375,-0.45%,-2.31%,-0.18%,-0.99%,0.36%,-4.02%
4,2026-07-14,CPI,Macro,2026-07-14 08:30,216.2018,2.20%,1.49%,0.80%,1.37%,1.92%,...,0.26%,0.19%,0.20%,84.6900,1.78%,1.26%,0.96%,0.77%,1.16%,-3.50%
5,2026-07-29,FOMC,Macro,2026-07-29 14:00,158.2800,2.51%,0.40%,0.32%,-1.60%,1.24%,...,0.18%,0.70%,-0.94%,63.2000,2.16%,1.56%,0.99%,0.08%,2.67%,-3.66%
6,2026-08-12,NBIS Q2 Earnings,Company,2026-08-12 08:00,220.2000,0.37%,1.21%,1.30%,1.88%,1.78%,...,-0.02%,0.17%,-0.01%,106.7100,0.22%,0.41%,0.35%,0.44%,0.44%,0.58%
7,2026-08-12,CPI,Macro,2026-08-12 08:30,225.2200,0.04%,0.35%,-0.01%,-0.49%,-2.78%,...,0.05%,0.05%,-0.10%,107.1000,1.26%,2.02%,1.06%,0.07%,1.71%,-0.15%


In [19]:
import numpy as np
import pandas as pd


# ============================================================
# 分析某个阻力位的每一次测试
# ============================================================

def analyze_resistance_attempts(
    daily_df,
    level=300,
    touch_pct=0.02,
    confirmation_pct=0.005,
    hold_days=3
):

    d = (
        daily_df
        .copy()
        .sort_index()
    )

    results = []


    # --------------------------------------------------------
    # 1. 找出进入阻力测试区域的交易日
    #
    # level = 300
    # touch_pct = 2%
    #
    # high >= 294
    # 就视为进入300美元阻力测试区域
    # --------------------------------------------------------

    touch_threshold = (
        level * (1 - touch_pct)
    )

    candidate_positions = np.where(
        d["high"].ge(touch_threshold)
    )[0]


    if len(candidate_positions) == 0:
        return pd.DataFrame()


    # --------------------------------------------------------
    # 2. 把连续交易日合并成同一轮测试
    # --------------------------------------------------------

    attempt_groups = [
        [candidate_positions[0]]
    ]

    for position in candidate_positions[1:]:

        if position == attempt_groups[-1][-1] + 1:

            attempt_groups[-1].append(
                position
            )

        else:

            attempt_groups.append(
                [position]
            )


    # --------------------------------------------------------
    # 3. 分析每一轮测试
    # --------------------------------------------------------

    for attempt_number, positions in enumerate(
        attempt_groups,
        start=1
    ):

        attempt = d.iloc[positions]

        start_position = positions[0]
        end_position = positions[-1]

        start_date = attempt.index[0]
        end_date = attempt.index[-1]


        # 本轮测试期间最高价
        peak_date = attempt["high"].idxmax()
        peak_high = attempt.loc[
            peak_date,
            "high"
        ]


        # 最高价所在的一分钟
        if "high_time" in attempt.columns:

            peak_time = attempt.loc[
                peak_date,
                "high_time"
            ]

        else:

            peak_time = pd.NaT


        # 距离阻力位的百分比
        #
        # -0.000467代表距离300还差0.0467%
        # 正数代表已经盘中超过300
        distance_from_level = (
            peak_high / level - 1
        )


        # 本轮最高收盘价
        max_close = (
            attempt["close"].max()
        )


        # 本轮最后一天收盘价
        attempt_end_close = (
            attempt.iloc[-1]["close"]
        )


        # 本轮是否盘中达到或突破阻力位
        crossed_intraday = (
            peak_high >= level
        )


        # 收盘确认线
        #
        # level=300
        # confirmation_pct=0.5%
        # 确认线就是301.50
        confirmation_level = (
            level
            * (1 + confirmation_pct)
        )


        # 是否曾经收盘明确站上确认线
        closed_above = (
            max_close
            >= confirmation_level
        )


        # ----------------------------------------------------
        # 4. 检查后续是否站稳
        # ----------------------------------------------------

        future_hold = d.iloc[
            end_position + 1:
            end_position + 1 + hold_days
        ]


        if len(future_hold) == hold_days:

            days_closed_above = (
                future_hold["close"]
                .gt(level)
                .sum()
            )

            held_above = (
                days_closed_above
                >= np.ceil(hold_days / 2)
            )

        else:

            held_above = False


        # ----------------------------------------------------
        # 5. 给本轮测试分类
        # ----------------------------------------------------

        if closed_above and held_above:

            status = "CONFIRMED BREAKOUT"

        elif crossed_intraday:

            status = "FALSE BREAKOUT"

        else:

            status = "FAILED ATTEMPT"


        # ----------------------------------------------------
        # 6. 计算测试结束后N个交易日收益
        # ----------------------------------------------------

        def forward_return(number_of_days):

            future_position = (
                end_position
                + number_of_days
            )

            if future_position >= len(d):
                return np.nan

            future_close = d.iloc[
                future_position
            ]["close"]

            return (
                future_close
                / attempt_end_close
                - 1
            )


        return_1d = forward_return(1)
        return_3d = forward_return(3)
        return_5d = forward_return(5)


        # ----------------------------------------------------
        # 7. 未来5个交易日最大回撤
        # ----------------------------------------------------

        future_5d = d.iloc[
            end_position + 1:
            end_position + 6
        ]


        if not future_5d.empty:

            future_low = (
                future_5d["low"].min()
            )

            max_drawdown_5d = (
                future_low
                / attempt_end_close
                - 1
            )

        else:

            max_drawdown_5d = np.nan


        # ----------------------------------------------------
        # 8. 测试期间最大RVOL
        # ----------------------------------------------------

        if "rvol20" in attempt.columns:

            max_rvol = (
                attempt["rvol20"].max()
            )

        else:

            max_rvol = np.nan


        # ----------------------------------------------------
        # 9. 保存这一轮测试结果
        # ----------------------------------------------------

        results.append({

            "attempt": attempt_number,

            "start_date": start_date,
            "end_date": end_date,

            "peak_date": peak_date,
            "peak_time": peak_time,

            "peak_high": peak_high,

            "distance_from_300":
                distance_from_level,

            "max_close": max_close,
            "attempt_end_close":
                attempt_end_close,

            "max_rvol": max_rvol,

            "crossed_300_intraday":
                crossed_intraday,

            "closed_above_confirmation":
                closed_above,

            "held_above_300":
                held_above,

            "return_1d": return_1d,
            "return_3d": return_3d,
            "return_5d": return_5d,

            "max_drawdown_5d":
                max_drawdown_5d,

            "status": status
        })


    return pd.DataFrame(results)

# test 300

In [20]:
import numpy as np
import pandas as pd


# ============================================================
# 分析某个阻力位的每一次测试
# ============================================================

def analyze_resistance_attempts(
    daily_df,
    level=300,
    touch_pct=0.02,
    confirmation_pct=0.005,
    hold_days=3
):

    d = (
        daily_df
        .copy()
        .sort_index()
    )

    results = []


    # --------------------------------------------------------
    # 1. 找出进入阻力测试区域的交易日
    #
    # level = 300
    # touch_pct = 2%
    #
    # high >= 294
    # 就视为进入300美元阻力测试区域
    # --------------------------------------------------------

    touch_threshold = (
        level * (1 - touch_pct)
    )

    candidate_positions = np.where(
        d["high"].ge(touch_threshold)
    )[0]


    if len(candidate_positions) == 0:
        return pd.DataFrame()


    # --------------------------------------------------------
    # 2. 把连续交易日合并成同一轮测试
    # --------------------------------------------------------

    attempt_groups = [
        [candidate_positions[0]]
    ]

    for position in candidate_positions[1:]:

        if position == attempt_groups[-1][-1] + 1:

            attempt_groups[-1].append(
                position
            )

        else:

            attempt_groups.append(
                [position]
            )


    # --------------------------------------------------------
    # 3. 分析每一轮测试
    # --------------------------------------------------------

    for attempt_number, positions in enumerate(
        attempt_groups,
        start=1
    ):

        attempt = d.iloc[positions]

        start_position = positions[0]
        end_position = positions[-1]

        start_date = attempt.index[0]
        end_date = attempt.index[-1]


        # 本轮测试期间最高价
        peak_date = attempt["high"].idxmax()
        peak_high = attempt.loc[
            peak_date,
            "high"
        ]


        # 最高价所在的一分钟
        if "high_time" in attempt.columns:

            peak_time = attempt.loc[
                peak_date,
                "high_time"
            ]

        else:

            peak_time = pd.NaT


        # 距离阻力位的百分比
        #
        # -0.000467代表距离300还差0.0467%
        # 正数代表已经盘中超过300
        distance_from_level = (
            peak_high / level - 1
        )


        # 本轮最高收盘价
        max_close = (
            attempt["close"].max()
        )


        # 本轮最后一天收盘价
        attempt_end_close = (
            attempt.iloc[-1]["close"]
        )


        # 本轮是否盘中达到或突破阻力位
        crossed_intraday = (
            peak_high >= level
        )


        # 收盘确认线
        #
        # level=300
        # confirmation_pct=0.5%
        # 确认线就是301.50
        confirmation_level = (
            level
            * (1 + confirmation_pct)
        )


        # 是否曾经收盘明确站上确认线
        closed_above = (
            max_close
            >= confirmation_level
        )


        # ----------------------------------------------------
        # 4. 检查后续是否站稳
        # ----------------------------------------------------

        future_hold = d.iloc[
            end_position + 1:
            end_position + 1 + hold_days
        ]


        if len(future_hold) == hold_days:

            days_closed_above = (
                future_hold["close"]
                .gt(level)
                .sum()
            )

            held_above = (
                days_closed_above
                >= np.ceil(hold_days / 2)
            )

        else:

            held_above = False


        # ----------------------------------------------------
        # 5. 给本轮测试分类
        # ----------------------------------------------------

        if closed_above and held_above:

            status = "CONFIRMED BREAKOUT"

        elif crossed_intraday:

            status = "FALSE BREAKOUT"

        else:

            status = "FAILED ATTEMPT"


        # ----------------------------------------------------
        # 6. 计算测试结束后N个交易日收益
        # ----------------------------------------------------

        def forward_return(number_of_days):

            future_position = (
                end_position
                + number_of_days
            )

            if future_position >= len(d):
                return np.nan

            future_close = d.iloc[
                future_position
            ]["close"]

            return (
                future_close
                / attempt_end_close
                - 1
            )


        return_1d = forward_return(1)
        return_3d = forward_return(3)
        return_5d = forward_return(5)


        # ----------------------------------------------------
        # 7. 未来5个交易日最大回撤
        # ----------------------------------------------------

        future_5d = d.iloc[
            end_position + 1:
            end_position + 6
        ]


        if not future_5d.empty:

            future_low = (
                future_5d["low"].min()
            )

            max_drawdown_5d = (
                future_low
                / attempt_end_close
                - 1
            )

        else:

            max_drawdown_5d = np.nan


        # ----------------------------------------------------
        # 8. 测试期间最大RVOL
        # ----------------------------------------------------

        if "rvol20" in attempt.columns:

            max_rvol = (
                attempt["rvol20"].max()
            )

        else:

            max_rvol = np.nan


        # ----------------------------------------------------
        # 9. 保存这一轮测试结果
        # ----------------------------------------------------

        results.append({

            "attempt": attempt_number,

            "start_date": start_date,
            "end_date": end_date,

            "peak_date": peak_date,
            "peak_time": peak_time,

            "peak_high": peak_high,

            "distance_from_300":
                distance_from_level,

            "max_close": max_close,
            "attempt_end_close":
                attempt_end_close,

            "max_rvol": max_rvol,

            "crossed_300_intraday":
                crossed_intraday,

            "closed_above_confirmation":
                closed_above,

            "held_above_300":
                held_above,

            "return_1d": return_1d,
            "return_3d": return_3d,
            "return_5d": return_5d,

            "max_drawdown_5d":
                max_drawdown_5d,

            "status": status
        })


    return pd.DataFrame(results)

In [21]:
# 先运行分析函数，生成原始结果
nbis_300_attempts = analyze_resistance_attempts(
    daily_df=nbis_daily,
    level=300,
    touch_pct=0.02,
    confirmation_pct=0.005,
    hold_days=3
)

# 创建用于显示的副本
nbis_300_display = nbis_300_attempts.copy()

In [22]:
# ============================================================
# 日期只显示年-月-日
# ============================================================

date_columns = [
    "start_date",
    "end_date",
    "peak_date"
]

for column in date_columns:

    nbis_300_display[column] = (
        pd.to_datetime(
            nbis_300_display[column]
        )
        .dt.strftime("%Y-%m-%d")
    )


# ============================================================
# 最高价时间只显示小时和分钟
# 不显示日期、秒和-04:00
# ============================================================

nbis_300_display["peak_time"] = (
    pd.to_datetime(
        nbis_300_display["peak_time"]
    )
    .dt.strftime("%H:%M")
)


# ============================================================
# 价格显示两位小数
# ============================================================

price_columns = [
    "peak_high",
    "max_close",
    "attempt_end_close"
]

for column in price_columns:

    nbis_300_display[column] = (
        nbis_300_display[column]
        .map(
            lambda value:
            f"${value:.2f}"
            if pd.notna(value)
            else ""
        )
    )


# ============================================================
# 百分比显示
# ============================================================

percent_columns = [
    "distance_from_300",
    "return_1d",
    "return_3d",
    "return_5d",
    "max_drawdown_5d"
]

for column in percent_columns:

    nbis_300_display[column] = (
        nbis_300_display[column]
        .map(
            lambda value:
            f"{value:.2%}"
            if pd.notna(value)
            else ""
        )
    )


# ============================================================
# RVOL显示为倍数
# ============================================================

nbis_300_display["max_rvol"] = (
    nbis_300_display["max_rvol"]
    .map(
        lambda value:
        f"{value:.2f}x"
        if pd.notna(value)
        else ""
    )
)


# ============================================================
# True/False显示为Yes/No
# ============================================================

boolean_columns = [
    "crossed_300_intraday",
    "closed_above_confirmation",
    "held_above_300"
]

for column in boolean_columns:

    nbis_300_display[column] = (
        nbis_300_display[column]
        .map({
            True: "Yes",
            False: "No"
        })
    )


# ============================================================
# 显示最重要的列
# ============================================================

display(
    nbis_300_display[
        [
            "attempt",
            "start_date",
            "end_date",
            "peak_date",
            "peak_time",
            "peak_high",
            "distance_from_300",
            "max_rvol",
            "return_1d",
            "return_3d",
            "return_5d",
            "max_drawdown_5d",
            "status"
        ]
    ]
)


,attempt,start_date,end_date,peak_date,peak_time,peak_high,distance_from_300,max_rvol,return_1d,return_3d,return_5d,max_drawdown_5d,status
0,1,2026-06-17,2026-06-23,2026-06-22,09:31,$299.86,-0.05%,1.52x,-5.71%,-12.68%,0.22%,-14.88%,FAILED ATTEMPT


In [23]:
import ctypes
from pathlib import Path

buf = ctypes.create_unicode_buffer(260)

ctypes.windll.shell32.SHGetFolderPathW(
    None,
    0x0000,   # CSIDL_DESKTOP
    None,
    0,
    buf
)

DESKTOP = Path(buf.value)

print("Real Desktop path:")
print(DESKTOP)
print("Exists:", DESKTOP.exists())

Real Desktop path:
C:\Users\rz321\OneDrive\Desktop
Exists: True


In [25]:
event_study.to_csv(
    DESKTOP / "NBIS_event_study.csv",
    index=False
)

nbis_daily.to_csv(
    DESKTOP / "NBIS_daily_price_action.csv"
)

bars["NBIS"].to_parquet(
    DESKTOP / "NBIS_1min_2026-05-01_2026-08-22.parquet"
)

print("Saved to:")
print(DESKTOP)

Saved to:
C:\Users\rz321\OneDrive\Desktop
